In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold

In [5]:
train = pd.read_csv("train.csv")
y = np.log1p(train["hours_spent"])
X = train.drop(columns=["hours_spent", "log_hours", "hours_bucket", "project_id"])

def add_features(df):
    df = df.copy()
    df["biggest_commit_share"] = df["max_lines_in_one_commit"] / df["lines_added_total"].replace(0, np.nan)
    df["lines_per_active_day"] = df["lines_of_code"] / df["active_commit_days"].replace(0, np.nan)
    df["commits_per_active_day"] = df["num_commits"] / df["active_commit_days"].replace(0, np.nan)
    df["active_day_ratio"] = df["active_commit_days"] / (df["project_span_days"] + 1)
    return df

In [6]:
ORDER_MAPS = {
    "author_prior_projects":     {"0": 0, "1-2": 1, "3-5": 2, "6+": 3},
    "author_account_age_years":  {"<1": 0, "1-2": 1, "3-5": 2, "6+": 3},
}
def encode_ordered(df):
    df = df.copy()
    for col, mapping in ORDER_MAPS.items():
        df[col] = df[col].astype(str).map(mapping)
    return df

In [7]:
def encode_categories(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("category")
    return df

In [8]:
def encode_yes_no(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        values = set(df[col].dropna().astype(str).unique())
        if values <= {"True", "False"}:
            df[col] = df[col].astype(str).map({"True": 1.0, "False": 0.0})
    return df

In [9]:
def prepare(df):
    df = add_features(df)
    df = encode_ordered(df)
    df = encode_yes_no(df)
    df = encode_categories(df)
    return df

X = prepare(X)

C:\Users\Fujitsu\AppData\Local\Temp\ipykernel_19688\1303781909.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\Fujitsu\AppData\Local\Temp\ipykernel_19688\3138021966.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guid

In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(X.shape)

(4500, 73)


In [11]:
from sklearn.metrics import root_mean_squared_log_error

def cv_score(make_model, X_data):
    oof = np.zeros(len(X_data))
    for train_idx, test_idx in kf.split(X_data):
        model = make_model()
        model.fit(X_data.iloc[train_idx], y.iloc[train_idx])
        oof[test_idx] = model.predict(X_data.iloc[test_idx])
    pred = np.expm1(oof).clip(0, 140.7)
    return root_mean_squared_log_error(train["hours_spent"], pred)

In [12]:
bool_cols = X.select_dtypes(include="bool").columns
X = X.astype({c: int for c in bool_cols})


cat_cols = X.select_dtypes(include="category").columns.tolist()
X_cat = X.copy()
for c in cat_cols:
    X_cat[c] = X_cat[c].astype(object).fillna("missing").astype(str)

In [13]:
import xgboost as xgb
from catboost import CatBoostRegressor

make_lgb = lambda: lgb.LGBMRegressor(
    n_estimators=800, learning_rate=0.03, num_leaves=31, min_child_samples=20,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, verbose=-1)

make_xgb = lambda: xgb.XGBRegressor(
    n_estimators=800, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    enable_categorical=True, tree_method="hist", random_state=42)

make_cat = lambda: CatBoostRegressor(
    iterations=1500, learning_rate=0.03, depth=6,
    cat_features=cat_cols, random_seed=42, verbose=0)

print("LightGBM:", round(cv_score(make_lgb, X), 4))
print("XGBoost: ", round(cv_score(make_xgb, X), 4))
print("CatBoost:", round(cv_score(make_cat, X_cat), 4))

LightGBM: 0.6643
XGBoost:  0.6613
CatBoost: 0.6607


In [14]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

d:\Xin Bing\VSC\hour-detective\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
def objective(trial):
    params = dict(
        n_estimators = trial.suggest_int("n_estimators", 300, 2000),
        learning_rate = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves = trial.suggest_int("num_leaves", 7, 63),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 100),
        subsample = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda = trial.suggest_float("reg_lambda", 0.001, 10, log=True),
    )
    make = lambda: lgb.LGBMRegressor(**params, subsample_freq=1, random_state=42, verbose=-1)
    return cv_score(make, X)

In [16]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best score: ", round(study.best_value, 4))
print("Best settings: ", study.best_params)

Best trial: 36. Best value: 0.655354: 100%|██████████| 50/50 [11:31<00:00, 13.83s/it]

Best score:  0.6554
Best settings:  {'n_estimators': 507, 'learning_rate': 0.01151459223519469, 'num_leaves': 63, 'min_child_samples': 27, 'subsample': 0.7569321677912477, 'colsample_bytree': 0.970087356836594, 'reg_lambda': 5.2049983097436945}


In [17]:
def cv_oof(make_model, X_data):
    oof = np.zeros(len(X_data))
    for train_idx, test_idx in kf.split(X_data):
        model = make_model()
        model.fit(X_data.iloc[train_idx], y.iloc[train_idx])
        oof[test_idx] = model.predict(X_data.iloc[test_idx])
    return oof

In [18]:
def score_from_oof(oof):
    pred = np.expm1(oof).clip(0, 140.7)
    return root_mean_squared_log_error(train["hours_spent"], pred)

In [19]:
best_params = study.best_params
make_lgb_best = lambda: lgb.LGBMRegressor(**best_params, subsample_freq=1, random_state=42, verbose=-1)

oof_lgb = cv_oof(make_lgb_best, X)
oof_xgb = cv_oof(make_xgb, X)
oof_cat = cv_oof(make_cat, X_cat)

print("LightGBM: ", round(score_from_oof(oof_lgb), 4))
print("XGBoost: ", round(score_from_oof(oof_xgb), 4))
print("CatBoost: ", round(score_from_oof(oof_cat), 4))

blend = (oof_lgb + oof_xgb + oof_cat) / 3
print("Average of all 3:", round(score_from_oof(blend), 4))

LightGBM:  0.6554
XGBoost:  0.6613
CatBoost:  0.6607
Average of all 3: 0.6524


In [20]:
pred = pd.read_csv("predictions.csv")
print(pred.shape)
print(pred.columns.tolist())
print(pred["hours_spent"].isna().sum())
print(pred["hours_spent"].describe())

(500, 2)
['project_id', 'hours_spent']
0
count    500.000000
mean       9.983733
std       12.205829
min        0.722754
25%        3.369277
50%        5.702204
75%       11.733667
max      103.599680
Name: hours_spent, dtype: float64


In [21]:
import xgboost as xgb
from catboost import CatBoostRegressor

BEST_LGB = {
    "n_estimators": 865, "learning_rate": 0.012964449902814406, "num_leaves": 35,
    "min_child_samples": 11, "subsample": 0.870297694890708,
    "colsample_bytree": 0.5828255385942684, "reg_lambda": 0.002402089003977252,
}

In [22]:
SEEDS = [42, 1, 2]

def make_lgb_seed(seed):
    return lgb.LGBMRegressor(**BEST_LGB, subsample_freq=1, random_state=seed, verbose=-1)

def make_xgb_seed(seed):
    return xgb.XGBRegressor(
        n_estimators=800, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        enable_categorical=True, tree_method="hist", random_state=seed
    )

def make_cat_seed(seed):
    return CatBoostRegressor(
        iterations=600, learning_rate=0.06, depth=6,
        cat_features=cat_cols, random_seed=seed, thread_count=-1,
        verbose=0, allow_writing_files=False
    )

def seed_avg_oof(make_with_seed, X_data):
    all_oofs = []
    for s in SEEDS:
        oof = cv_oof(lambda: make_with_seed(s), X_data)
        all_oofs.append(oof)
        print(f"seed {s}: {score_from_oof(oof):.4f}")
    return np.mean(all_oofs, axis=0)

print("LightGBM"); oof_lgb_s = seed_avg_oof(make_lgb_seed, X)
print("XGBoost"); oof_xgb_s = seed_avg_oof(make_xgb_seed, X)
print("CatBoost"); oof_cat_s = seed_avg_oof(make_cat_seed, X_cat)

print()
print("LightGBM (3 seeds): ", round(score_from_oof(oof_lgb_s), 4))
print("XGBoost (3 seeds): ", round(score_from_oof(oof_xgb_s), 4))
print("CatBoost (3 seeds): ", round(score_from_oof(oof_cat_s), 4))

blend_s = (oof_lgb_s + oof_xgb_s + oof_cat_s) / 3
print("Blend (3 seeds): ", round(score_from_oof(blend_s), 4), " <- compare with 0.6532")

LightGBM
seed 42: 0.6554
seed 1: 0.6569
seed 2: 0.6584
XGBoost
seed 42: 0.6613
seed 1: 0.6611
seed 2: 0.6602
CatBoost
seed 42: 0.6672
seed 1: 0.6638
seed 2: 0.6648

LightGBM (3 seeds):  0.6557
XGBoost (3 seeds):  0.6578
CatBoost (3 seeds):  0.6619
Blend (3 seeds):  0.6536  <- compare with 0.6532


In [24]:
def make_cat_seed_slow(seed):
    return CatBoostRegressor(
        iterations=1500, learning_rate=0.03, depth=6,
        cat_features=cat_cols, random_seed=seed, thread_count=-1,
        verbose=0, allow_writing_files=False
    )

print("CatBoost slow"); oof_cat_slow = seed_avg_oof(make_cat_seed_slow, X_cat)

blend_best = (oof_lgb_s + oof_xgb_s + oof_cat_slow) / 3
print("CatBoost slow (3 seeds): ", round(score_from_oof(oof_cat_slow), 4))
print("Blend (3 seeds, slow Cat): ", round(score_from_oof(blend_best), 4))

CatBoost slow
seed 42: 0.6607
seed 1: 0.6624
seed 2: 0.6630
CatBoost slow (3 seeds):  0.66
Blend (3 seeds, slow Cat):  0.6533


In [25]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_xgb(trial):
    params = dict(
        n_estimators = trial.suggest_int("n_estimators", 300, 1500),
        learning_rate = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        max_depth = trial.suggest_int("max_depth", 3, 10),
        min_child_weight = trial.suggest_float("min_child_weight", 1, 20, log=True),
        subsample = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda = trial.suggest_float("reg_lambda", 0.001, 10, log=True),
    )
    make = lambda: xgb.XGBRegressor(**params, enable_categorical=True,
                                    tree_method="hist", random_state=42)
    return cv_score(make, X)

study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

Best trial: 26. Best value: 0.653646: 100%|██████████| 30/30 [12:19<00:00, 24.66s/it]


In [26]:
print("Old XGBoost: 0.6613")
print("Best score: ", round(study_xgb.best_value, 4))
print("Best settings: ", study_xgb.best_params)

Old XGBoost: 0.6613
Best score:  0.6536
Best settings:  {'n_estimators': 814, 'learning_rate': 0.010311069581219252, 'max_depth': 9, 'min_child_weight': 7.614778905496447, 'subsample': 0.5864764800801583, 'colsample_bytree': 0.3503811143222532, 'reg_lambda': 0.024691999110864553}


In [28]:
BEST_XGB = study_xgb.best_params

def make_xgb_tuned_seed(seed):
    return xgb.XGBRegressor(**BEST_XGB, enable_categorical=True,
                            tree_method="hist", random_state=seed)

print("XGBoost tuned"); oof_xgb_t = seed_avg_oof(make_xgb_tuned_seed, X)

blend_t = (oof_lgb_s + oof_xgb_t + oof_cat_s) / 3
print()
print("XGBoost old   (3 seeds): 0.6578")
print("XGBoost tuned (3 seeds):", round(score_from_oof(oof_xgb_t), 4))
print("Blend old     (3 seeds): 0.6536")
print("Blend tuned   (3 seeds):", round(score_from_oof(blend_t), 4))
print()
print("BEST_XGB =", BEST_XGB)

XGBoost tuned
seed 42: 0.6536
seed 1: 0.6539
seed 2: 0.6550

XGBoost old   (3 seeds): 0.6578
XGBoost tuned (3 seeds): 0.6525
Blend old     (3 seeds): 0.6536
Blend tuned   (3 seeds): 0.6533

BEST_XGB = {'n_estimators': 814, 'learning_rate': 0.010311069581219252, 'max_depth': 9, 'min_child_weight': 7.614778905496447, 'subsample': 0.5864764800801583, 'colsample_bytree': 0.3503811143222532, 'reg_lambda': 0.024691999110864553}


In [30]:
teams = {
    "XGB tuned alone": oof_xgb_t,
    "LGB + XGB": (oof_lgb_s + oof_xgb_t) / 2,
    "LGB + XGB + Cat (equal)": (oof_lgb_s + oof_xgb_t + oof_cat_s) / 3,
    "0.5 XGB + 0.3 LGB + 0.2 Cat": 0.5 * oof_xgb_t + 0.3 * oof_lgb_s + 0.2 * oof_cat_s,
}

for name, oof in teams.items():
    print(f"{name:.30s} {score_from_oof(oof):.4f}")

XGB tuned alone 0.6525
LGB + XGB 0.6527
LGB + XGB + Cat (equal) 0.6533
0.5 XGB + 0.3 LGB + 0.2 Cat 0.6524


In [31]:
best_oof = 0.5 * oof_xgb_t + 0.3 * oof_lgb_s + 0.2 * oof_cat_s

err = pd.DataFrame({
    "real_hours": train["hours_spent"],
    "pred_hours": np.expm1(best_oof).clip(0, 140.7).round(1),
})

err["error"] = best_oof - y
err["times_off"] = np.exp(err["error"].abs()).round(1)

show = ["real_hours", "pred_hours", "times_off",
        "active_commit_days", "project_span_days", "num_commits",
        "lines_of_code", "pct_code_in_first_commit", "readme_word_count",
        "project_category", "url_is_live"]

info = pd.concat([err, train.drop(columns=["hours_spent"])], axis=1)

print("Guessed Too High (real hours were small): ")
display(info.sort_values("error", ascending=False)[show].head(10))

print("Guessed Too Low (real hours were big): ")
display(info.sort_values("error")[show].head(10))

Guessed Too High (real hours were small): 


,real_hours,pred_hours,times_off,active_commit_days,project_span_days,num_commits,lines_of_code,pct_code_in_first_commit,readme_word_count,project_category,url_is_live
1779,1.0,37.7,19.4,35,175,62,18100,0.0055,0,game,True
9,0.3,16.2,13.2,11,42,21,1900,0.1452,461,tool,True
906,0.2,13.6,12.1,4,3,12,500,0.7610,441,app,True
3606,1.2,23.1,11.0,159,437,325,4600,0.0039,348,other,False
2117,0.8,18.4,10.8,2,370,5,1300,0.7172,273,app,True
3707,0.5,14.7,10.5,16,485,58,27500,0.0040,575,app,True
863,0.6,15.8,10.5,34,385,76,2200,0.2521,67,app,False
2850,1.0,18.2,9.6,400,848,1214,14700,0.0008,260,other,False
4416,2.3,27.1,8.5,18,17,48,2000,0.1070,13,game,False
3873,0.2,9.2,8.5,48,224,273,2100,0.0628,195,website,True


Guessed Too Low (real hours were big): 


,real_hours,pred_hours,times_off,active_commit_days,project_span_days,num_commits,lines_of_code,pct_code_in_first_commit,readme_word_count,project_category,url_is_live
3372,140.7,6.1,19.9,8,565,18,0,NaN,6,tool,True
1245,56.0,2.0,19.1,1,0,1,0,1.0000,161,website,False
248,140.7,6.6,18.5,1,0,1,37400,1.0000,8718,game,True
8,53.5,2.2,16.9,3,623,27,0,0.6686,60,bot,True
2084,140.7,7.5,16.6,603,3399,1701,34000,0.0022,648,tool,True
2505,140.7,8.6,14.8,89,981,171,3100,0.0318,34,website,False
1217,140.7,8.8,14.5,8,356,15,8300,0.6014,549,app,False
1557,101.2,6.2,14.2,2,2,12,400,0.0747,7,game,True
2104,48.7,2.8,13.2,4,1288,20,200,0.0325,40,app,True
2793,60.0,4.5,11.0,3,113,4,0,1.0000,39,app,True


In [33]:
low_cut, high_cut = err["error"].quantile([0.10,0.90])
group = np.where(err["error"] >= high_cut, "too high",
                 np.where(err["error"] <= low_cut, "too low", "ok"))

num_cols = X.select_dtypes(include="number").columns
Z = (X[num_cols] - X[num_cols].mean()) / X[num_cols].std()

diff = Z.groupby(group).mean().T
diff["high_minus_ok"] = diff["too high"] - diff["ok"]
diff["low_minus_ok"] = diff["too low"] - diff["ok"]

print("🔼 What makes 'too high' projects different:")
print(diff["high_minus_ok"].sort_values(key=abs, ascending=False).head(10).round(2))
print()
print("🔽 What makes 'too low' projects different:")
print(diff["low_minus_ok"].sort_values(key=abs, ascending=False).head(10).round(2))
print()
print("Project types in each group (share of projects):")
display(pd.crosstab(group, train["project_category"], normalize="index").round(2))

🔼 What makes 'too high' projects different:
active_day_ratio      -0.43
readme_num_headings    0.42
project_span_days      0.36
active_commit_days     0.33
max_dir_depth          0.32
churn_ratio            0.31
stars                  0.31
readme_word_count      0.29
num_languages          0.27
num_dependencies       0.27
Name: high_minus_ok, dtype: float64

🔽 What makes 'too low' projects different:
stars                     0.38
active_day_ratio         -0.37
lines_of_code             0.36
active_commit_days        0.35
num_source_files          0.33
project_span_days         0.32
readme_word_count         0.31
readme_num_headings       0.30
max_dir_depth             0.29
description_word_count    0.29
Name: low_minus_ok, dtype: float64

Project types in each group (share of projects):


project_category,app,bot,game,other,tool,website
row_0,,,,,,
ok,0.20,0.03,0.40,0.02,0.15,0.21
too high,0.30,0.04,0.27,0.03,0.16,0.21
too low,0.24,0.03,0.30,0.02,0.20,0.20
